# Exercise 70 — Mini-project: CSV deduplicator (latest-wins)

## Problem

Upstream sends you a CSV of customer state changes. Multiple rows can share the same `customer_id` — each one is a different snapshot in time, with a `ts` column. You need a single row per customer: **the most recent** by `ts`.

```
customer_id,ts,name,city
1,2026-01-01T08:00,Alice,Curitiba
2,2026-01-01T09:30,Bob,Rio
1,2026-01-03T12:00,Alice,Curitiba
3,2026-01-02T10:00,Carol,SP
1,2026-01-02T15:00,Alicia,Curitiba       ← supersedes the 08:00 row
2,2026-01-01T09:30,Bob,Rio              ← exact duplicate of row 2, keep one
```

Expected output: 3 rows total (one per customer), with the latest snapshot.

## Two implementations

Do it both in **pure Python** (dict-keyed by id, replace if newer) and **pandas** (groupby + idxmax). The pure-Python version is what you'd write when the source is a streaming gzipped CSV that doesn't fit in memory; pandas is faster when it fits.

## Setup

In [ ]:
from pathlib import Path

CSV = "customers.csv"
Path(CSV).write_text(
    "customer_id,ts,name,city\n"
    "1,2026-01-01T08:00,Alice,Curitiba\n"
    "2,2026-01-01T09:30,Bob,Rio\n"
    "1,2026-01-03T12:00,Alice,Curitiba\n"
    "3,2026-01-02T10:00,Carol,SP\n"
    "1,2026-01-02T15:00,Alicia,Curitiba\n"
    "2,2026-01-01T09:30,Bob,Rio\n",
    encoding="utf-8",
)
print("setup ok")


## Task 1 — Pure Python (streaming-friendly)

Write a function that returns one row per `customer_id`, keeping the row with the **maximum `ts`** (ISO strings compare correctly as text, so no parsing needed).

```python
def dedupe_latest(csv_path: str) -> list[dict]:
    """Return the deduplicated rows as a list of dicts (DictReader-style).
    Result order: sorted by customer_id ascending."""
```

Approach:
- Walk the CSV with `DictReader`.
- Keep a dict `latest_by_id: dict[str, dict]`.
- For each row, replace the existing entry only if the new `ts` is greater (or no entry exists yet).
- At the end, return values sorted by `int(customer_id)`.

Expected:
```
[
    {"customer_id": "1", "ts": "2026-01-03T12:00", "name": "Alice",  "city": "Curitiba"},
    {"customer_id": "2", "ts": "2026-01-01T09:30", "name": "Bob",    "city": "Rio"},
    {"customer_id": "3", "ts": "2026-01-02T10:00", "name": "Carol",  "city": "SP"},
]
```

In [ ]:
import csv

def dedupe_latest(csv_path: str) -> list[dict]:
    pass  # TODO

result = dedupe_latest(CSV)
assert result == [
    {"customer_id": "1", "ts": "2026-01-03T12:00", "name": "Alice",  "city": "Curitiba"},
    {"customer_id": "2", "ts": "2026-01-01T09:30", "name": "Bob",    "city": "Rio"},
    {"customer_id": "3", "ts": "2026-01-02T10:00", "name": "Carol",  "city": "SP"},
]
print("ok")


## Task 2 — pandas version

Do the same dedup with pandas. Return a DataFrame sorted by `customer_id` ascending.

```python
def dedupe_latest_pd(csv_path: str) -> pd.DataFrame:
    ...
```

Approach: read the CSV, sort by `[customer_id, ts]`, then `drop_duplicates("customer_id", keep="last")` and `reset_index(drop=True)`.

Alternative: groupby + `idxmax`. Either is fine.

In [ ]:
import pandas as pd

def dedupe_latest_pd(csv_path: str) -> pd.DataFrame:
    pass  # TODO

result = dedupe_latest_pd(CSV)
assert result["customer_id"].tolist() == [1, 2, 3]
assert result.loc[result["customer_id"] == 1, "name"].iloc[0] == "Alice"     # latest = 2026-01-03 row
assert result.loc[result["customer_id"] == 1, "ts"].iloc[0] == "2026-01-03T12:00"
assert result.loc[result["customer_id"] == 3, "name"].iloc[0] == "Carol"
print("ok")


## Task 3 — Write the dedup result back to a CSV

Use either implementation. Write `customers_dedup.csv` with the same column order as the input.

```python
def write_dedup_csv(src: str, dst: str) -> int:
    """Return number of rows written (excluding header)."""
```

In [ ]:
import csv
from pathlib import Path

def write_dedup_csv(src: str, dst: str) -> int:
    pass  # TODO

n = write_dedup_csv(CSV, "customers_dedup.csv")
assert n == 3

lines = Path("customers_dedup.csv").read_text(encoding="utf-8").splitlines()
assert lines[0] == "customer_id,ts,name,city"
assert len(lines) - 1 == 3   # 3 data rows
print("ok")
